# Level 2 — Feature Engineering with DuckDB

## Objective

This notebook loads the cleaned dataset into DuckDB and creates engineered features that support downstream business analysis. These features enrich the original dataset by deriving new metrics, validating their correctness, and preparing the data for more advanced analytics in the next notebook.

## Load Dataset into DuckDB     

Load the cleaned dataset into DuckDB to begin feature engineering and analytical processing. 

In [1]:
from pathlib import Path
import duckdb

# Path to the cleaned dataset
data_path = Path("../data/superstore_utf8.csv")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE VIEW superstore AS
SELECT *
FROM read_csv_auto('{data_path.as_posix()}', header=True)
""")

con.execute("""
SELECT *
FROM superstore
LIMIT 5
""").df()

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


## Feature Engineering

### Feature Creation

A DuckDB view named `superstore_features` was created to preserve the original dataset while adding analytical features. New columns include `fulfillment_days`, `profit_margin`, `order_year`, `order_month`, and `customer_lifetime_sales` (calculated using a window function). These engineered features enrich the dataset with business-focused metrics for downstream customer, sales, profitability, and operational analysis.

In [2]:
con.execute("""
CREATE OR REPLACE VIEW superstore_features AS
SELECT
    *,
    date_diff('day', order_date, ship_date) AS fulfillment_days,
    profit / sales AS profit_margin,
    year(order_date) AS order_year,
    month(order_date) AS order_month,
    SUM(sales) OVER (
        PARTITION BY customer_id
    ) AS customer_lifetime_sales
FROM superstore
""") 

### Inspecting Engineered Features

The newly created features were queried from the `superstore_features` view and inspected using a sample of 10 records. This validation step ensured that the feature engineering process produced the expected values before proceeding with further analysis.

In [3]:
con.execute("""
SELECT
    order_id,
    fulfillment_days,
    profit_margin,
    order_year,
    order_month,
    customer_lifetime_sales
FROM superstore_features
ORDER BY order_date
LIMIT 10;
""").df()

,order_id,fulfillment_days,profit_margin,order_year,order_month,customer_lifetime_sales
0,CA-2014-103800,4,0.3375,2014,1,1050.636
1,CA-2014-112326,4,-0.2375,2014,1,1056.858
2,CA-2014-112326,4,-1.5500,2014,1,1056.858
3,CA-2014-112326,4,0.3625,2014,1,1056.858
4,CA-2014-141817,7,0.2500,2014,1,1428.231
5,CA-2014-167199,4,0.2700,2014,1,10663.728
6,CA-2014-167199,4,0.2900,2014,1,10663.728
7,CA-2014-167199,4,0.2900,2014,1,10663.728
8,CA-2014-167199,4,0.0100,2014,1,10663.728
9,CA-2014-167199,4,0.4500,2014,1,10663.728


## Feature Validation

Each engineered feature was validated to ensure the calculations produced reasonable and expected results before being used in downstream business analysis.

### Fulfillment Days Validation

The distribution of the engineered `fulfillment_days` feature was examined to verify that the calculated values were reasonable. Most orders were fulfilled within **4–5 days**, with relatively few orders requiring **0–1 days** or the maximum of **7 days**, indicating a realistic distribution suitable for downstream operational analysis.

In [4]:
con.execute('''
SELECT
    fulfillment_days,
    COUNT(*) AS orders
FROM superstore_features
GROUP BY fulfillment_days
ORDER BY fulfillment_days;
''').df().style.hide(axis="index")            

fulfillment_days,orders
0,519
1,369
2,1334
3,1005
4,2774
5,2169
6,1203
7,621


### Profit Margin Validation

The engineered `profit_margin` feature, calculated as `profit / sales`, represents the proportion of each sale retained as profit. Analysis showed an average profit margin of **12.03%**, indicating that the company earned approximately **12 cents of profit for every dollar of sales**. Profit margins ranged from **-275%** to **50%**, highlighting that while some transactions were highly profitable, others resulted in substantial losses.

In [5]:
df = con.execute("""
SELECT
    ROUND(AVG(profit_margin), 2) AS avg_profit_margin,
    ROUND(MIN(profit_margin), 2) AS min_margin,
    ROUND(MEDIAN(profit_margin), 2) AS median_margin,
    ROUND(MAX(profit_margin), 2) AS max_margin
FROM superstore_features;
""").df()

display(df.style.hide(axis="index").format("{:g}"))

avg_profit_margin,min_margin,median_margin,max_margin
0.12,-2.75,0.27,0.5


**Summary:**

The products with the lowest profit margins were identified to better understand the transactions contributing to overall losses. Several products exhibited profit margins below **-270%**, indicating that the losses incurred on these sales substantially exceeded the revenue generated. These extreme values confirm that the engineered `profit_margin` feature successfully captures both profitable and loss-making transactions and is suitable for downstream profitability analysis.

### Order Year Validation 

The engineered `order_year` feature was validated by summarizing the number of orders placed in each year.

In [6]:
con.execute('''
SELECT
    order_year,
    COUNT(*) AS orders
FROM superstore_features
GROUP BY order_year
ORDER BY order_year;
''').df().style.hide(axis="index")

order_year,orders
2014,1993
2015,2102
2016,2587
2017,3312


### Order Month Validation 

The engineered `order_month` feature was validated by summarizing the number of orders placed in each month.

In [7]:
con.execute('''
SELECT
    order_month,
    COUNT(*) AS orders
FROM superstore_features
GROUP BY order_month
ORDER BY order_month;
''').df().style.hide(axis="index")

order_month,orders
1,381
2,300
3,696
4,668
5,735
6,717
7,710
8,706
9,1383
10,819


### Customer Lifetime Sales Validation

The engineered `customer_lifetime_sales` feature was validated by identifying the highest-value customers in the dataset. Sean Miller generated over **$25,000** in lifetime sales, while several other customers exceeded **$12,000**, confirming that the window function correctly calculated customer-level sales across multiple transactions.

In [8]:
con.execute('''
SELECT DISTINCT
    customer_name,
    customer_lifetime_sales
FROM superstore_features
ORDER BY customer_lifetime_sales DESC
LIMIT 10;
''').df().style.hide(axis="index")

customer_name,customer_lifetime_sales
Sean Miller,25043.050000
Tamara Chand,19052.218000
Raymond Buch,15117.339000
Tom Ashbrook,14595.620000
Adrian Barton,14473.571000
Ken Lonsdale,14175.229000
Sanjit Chand,14142.334000
Hunter Lopez,12873.298000
Sanjit Engle,12209.438000
Christopher Conant,12129.072000


### Summary

All engineered features were successfully validated using descriptive statistics, frequency summaries, and sample record inspections. The results confirmed that each feature was calculated correctly and provided meaningful analytical value before additional business-oriented features were introduced.

## Customer Value Classification

Customer lifetime sales were further transformed into a categorical feature to simplify the identification of high-value customers. This business-oriented classification enables more intuitive customer comparisons in downstream analyses.

### Determine Classification Thresholds

Percentiles of the `customer_lifetime_sales` distribution were calculated to establish data-driven thresholds for customer value classification. Using these statistical breakpoints ensures that customer tiers are based on the underlying distribution of the data rather than arbitrary spending thresholds.

In [9]:
con.execute('''
SELECT
    quantile_cont(customer_lifetime_sales, 0.25) AS q1,
    quantile_cont(customer_lifetime_sales, 0.50) AS median,
    quantile_cont(customer_lifetime_sales, 0.75) AS q3,
    quantile_cont(customer_lifetime_sales, 0.9) AS top_10,
FROM (
    SELECT DISTINCT
        customer_id,
        customer_lifetime_sales
    FROM superstore_features
);
''').df().style.hide(axis="index")

q1,median,q3,top_10
1146.050000,2256.394000,3785.276000,6038.480000


### Create the `customer_tier` Feature

A new feature, `customer_tier`, was engineered by applying the previously calculated percentile thresholds to each customer's lifetime sales. Using a `CASE` statement, customers were classified into Standard, High Value, Premium, and Big Fish tiers, creating a reusable business feature for downstream customer analysis.

In [10]:
con.execute("""
CREATE OR REPLACE VIEW superstore_features AS
SELECT
    *,
    CASE
        WHEN customer_lifetime_sales >= 6038.48 THEN 'Big Fish'
        WHEN customer_lifetime_sales >= 3785.276 THEN 'Premium'
        WHEN customer_lifetime_sales >= 2256.394 THEN 'High Value'
        ELSE 'Standard'
    END AS customer_tier
FROM (
    SELECT
        *,
        date_diff('day', order_date, ship_date) AS fulfillment_days,
        profit / sales AS profit_margin,
        year(order_date) AS order_year,
        month(order_date) AS order_month,
        SUM(sales) OVER (
            PARTITION BY customer_id
        ) AS customer_lifetime_sales
    FROM superstore
);
""")

### Customer Tier Validation

A sample of customers and their corresponding `customer_lifetime_sales` and `customer_tier` values was reviewed to verify that the classification logic was applied correctly. The results confirmed that customers with higher lifetime sales were appropriately assigned to higher value tiers.

In [11]:
con.execute('''
SELECT DISTINCT
        segment,
        customer_name,
        customer_lifetime_sales,
        customer_tier
FROM superstore_features
ORDER BY customer_lifetime_sales DESC
LIMIT 10;
''').df().style.hide(axis="index")

segment,customer_name,customer_lifetime_sales,customer_tier
Home Office,Sean Miller,25043.050000,Big Fish
Corporate,Tamara Chand,19052.218000,Big Fish
Consumer,Raymond Buch,15117.339000,Big Fish
Home Office,Tom Ashbrook,14595.620000,Big Fish
Consumer,Adrian Barton,14473.571000,Big Fish
Consumer,Ken Lonsdale,14175.229000,Big Fish
Consumer,Sanjit Chand,14142.334000,Big Fish
Consumer,Hunter Lopez,12873.298000,Big Fish
Consumer,Sanjit Engle,12209.438000,Big Fish
Consumer,Christopher Conant,12129.072000,Big Fish


### Summary

The `customer_tier` feature was successfully engineered and validated using percentile-based thresholds. By converting continuous lifetime sales into meaningful customer categories, the dataset now includes a reusable business feature for downstream customer segmentation and reporting.

## Final Engineered Dataset

The completed `superstore_features` view was inspected to verify that all engineered features were successfully incorporated into a single analytical dataset. This final validation confirms that the dataset is complete, internally consistent, and ready for SQL-based business analysis in the next notebook.

In [12]:
con.execute('''
SELECT
    order_id,
    order_year,
    order_month,
    fulfillment_days,
    profit_margin,
    customer_lifetime_sales,
    customer_tier
FROM superstore_features
ORDER BY order_id
LIMIT 10;
''').df().style.hide(axis="index")

order_id,order_year,order_month,fulfillment_days,profit_margin,customer_lifetime_sales,customer_tier
CA-2014-100006,2014,9,6,0.290000,3318.486000,High Value
CA-2014-100090,2014,7,4,-0.175000,3644.978000,High Value
CA-2014-100090,2014,7,4,0.350000,3644.978000,High Value
CA-2014-100293,2014,3,4,0.350000,377.162000,Standard
CA-2014-100328,2014,1,6,0.337500,71.263000,Standard
CA-2014-100363,2014,4,7,0.362500,864.947000,Standard
CA-2014-100363,2014,4,7,0.350000,864.947000,Standard
CA-2014-100391,2014,5,4,0.460000,385.516000,Standard
CA-2014-100678,2014,4,4,0.337500,4909.472000,Premium
CA-2014-100678,2014,4,4,0.125000,4909.472000,Premium


## Notebook Summary

This notebook extended the cleaned Superstore dataset by engineering new analytical features using DuckDB. Date functions, arithmetic calculations, window functions, aggregate functions, and conditional logic were used to create reusable business metrics that provide deeper insight into customer behavior, operational performance, and profitability.

Each engineered feature was validated to ensure the calculations were accurate and produced meaningful analytical information before being used in downstream analysis. By separating feature engineering from business analysis, this project establishes a reliable analytical foundation that will be leveraged in the next notebook to answer key business questions and generate actionable insights.

---

## Key Takeaways

- Engineered reusable analytical features using DuckDB SQL.
- Applied date functions, arithmetic calculations, window functions, aggregate functions, and conditional logic to create business-focused metrics.
- Validated each engineered feature using descriptive statistics, frequency summaries, and sample record inspections.
- Created a business-oriented `customer_tier` feature using data-driven percentile thresholds.
- Produced the reusable `superstore_features` view to support subsequent SQL-based business analysis.

---

## Next Steps

The next notebook will use the `superstore_features` view to answer key business questions related to customer value, sales performance, profitability, and operational efficiency through SQL-based analytical queries. Building upon the engineered features created in this notebook, the analysis will identify meaningful business trends and demonstrate how SQL can be used to generate actionable insights from transactional data.